In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 1. Import Libraries and Tools

In [ ]:
import os
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageEnhance
from sklearn.utils import shuffle
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Flatten, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import EarlyStopping

print('✅ All libraries imported successfully!')

# 2. Configuration & Dataset Paths

In [ ]:
# Update these paths to match YOUR dataset location
TRAIN_DIR = '/content/drive/MyDrive/CV/MRI_Images/Training/'  # Update if needed
TEST_DIR  = '/content/drive/MyDrive/CV/MRI_Images/Testing/'   # Update if needed

IMAGE_SIZE = 128
BATCH_SIZE = 32      # Increased from 20 for stable training
EPOCHS = 20          # Increased from 5 for proper learning
LEARNING_RATE = 0.00005  # Decreased from 0.0001 for gentle fine-tuning

print(f"Configuration:")
print(f"  Train Directory: {TRAIN_DIR}")
print(f"  Test Directory : {TEST_DIR}")
print(f"  Image Size     : {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"  Batch Size     : {BATCH_SIZE}")
print(f"  Epochs         : {EPOCHS}")
print(f"  Learning Rate  : {LEARNING_RATE}")

# 3. ⚠️ CRITICAL FIX: Explicit Class Mapping
This fixes the 'every prediction is wrong' bug by using deterministic class ordering.

In [ ]:
# KEY FIX: Define class order ONCE using sorted() - deterministic and consistent!
# This prevents os.listdir() from returning different orders on different calls

CLASS_FOLDERS = sorted(os.listdir(TRAIN_DIR))
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASS_FOLDERS)}
IDX_TO_CLASS = {idx: cls for idx, cls in enumerate(CLASS_FOLDERS)}

print("="*70)
print("CLASS MAPPING (Fixed - This is what model uses)")
print("="*70)
for idx, cls in IDX_TO_CLASS.items():
    print(f"  Index {idx} ← → {cls:15} (class name)")
print("="*70)
print("\n✅ Class mapping is EXPLICIT and DETERMINISTIC")

# 4. Load Dataset Paths and Labels

In [ ]:
def load_dataset_paths(data_dir):
    """Load image paths and labels from directory structure."""
    paths = []
    labels = []
    for class_name in os.listdir(data_dir):
        class_folder = os.path.join(data_dir, class_name)
        if not os.path.isdir(class_folder):
            continue
        for img_file in os.listdir(class_folder):
            full_path = os.path.join(class_folder, img_file)
            paths.append(full_path)
            labels.append(class_name)
    
    paths, labels = shuffle(paths, labels)
    return paths, labels

# Load both training and testing datasets
train_paths, train_labels = load_dataset_paths(TRAIN_DIR)
test_paths, test_labels = load_dataset_paths(TEST_DIR)

print(f"✅ Training samples : {len(train_paths)}")
print(f"✅ Testing  samples : {len(test_paths)}")
print(f"✅ Classes found    : {CLASS_FOLDERS}")

# 5. Data Visualization - Check Sample Images

In [ ]:
# Visualize random training images
random_indices = random.sample(range(len(train_paths)), min(10, len(train_paths)))
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.ravel()

for i, idx in enumerate(random_indices):
    try:
        img = Image.open(train_paths[idx]).resize((IMAGE_SIZE, IMAGE_SIZE))
        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(train_labels[idx], fontsize=10, fontweight='bold')
    except Exception as e:
        print(f"Error loading image: {e}")

plt.suptitle("Sample Training Images", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 6. Image Preprocessing & Enhanced Augmentation

In [ ]:
def augment_image_improved(image):
    """Enhanced augmentation with rotation, flip, zoom."""
    pil_img = Image.fromarray(np.uint8(image))
    
    # Random brightness (wider range)
    pil_img = ImageEnhance.Brightness(pil_img).enhance(random.uniform(0.7, 1.3))
    
    # Random contrast
    pil_img = ImageEnhance.Contrast(pil_img).enhance(random.uniform(0.7, 1.3))
    
    # Random rotation (±15 degrees)
    if random.random() > 0.5:
        angle = random.uniform(-15, 15)
        pil_img = pil_img.rotate(angle, expand=False, fillcolor=0)
    
    # Random horizontal flip
    if random.random() > 0.5:
        pil_img = pil_img.transpose(Image.FLIP_LEFT_RIGHT)
    
    # Random zoom
    if random.random() > 0.5:
        w, h = pil_img.size
        zoom_factor = random.uniform(0.85, 1.0)
        new_size = int(w * zoom_factor)
        left = (w - new_size) // 2
        top = (h - new_size) // 2
        pil_img = pil_img.crop((left, top, left + new_size, top + new_size))
        pil_img = pil_img.resize((w, h))
    
    return np.array(pil_img) / 255.0


def open_images(paths):
    """Load and augment images."""
    images = []
    for path in paths:
        img = load_img(path, target_size=(IMAGE_SIZE, IMAGE_SIZE))
        img = augment_image_improved(img)
        images.append(img)
    return np.array(images)


def encode_labels(labels):
    """Convert class names to indices using EXPLICIT mapping."""
    return np.array([CLASS_TO_IDX[lbl] for lbl in labels])


# Test the encoding
sample_labels = ['glioma', 'notumor', 'meningioma', 'pituitary'] if 'glioma' in CLASS_FOLDERS else CLASS_FOLDERS
sample_labels = sample_labels[:min(4, len(sample_labels))]
encoded = encode_labels(sample_labels)
decoded = [IDX_TO_CLASS[i] for i in encoded]

print("Encoding Test:")
print(f"  Original: {sample_labels}")
print(f"  Encoded : {encoded}")
print(f"  Decoded : {decoded}")
print(f"  Status  : {'✅ CORRECT' if sample_labels == decoded else '❌ WRONG'}")

# 7. Data Generator (Memory-Efficient Batching)

In [ ]:
def data_generator(paths, labels, batch_size=BATCH_SIZE, epochs=1):
    """Yields batches of images with proper label encoding."""
    for epoch in range(epochs):
        indices = np.arange(len(paths))
        np.random.shuffle(indices)
        
        for start_idx in range(0, len(paths), batch_size):
            batch_indices = indices[start_idx : start_idx + batch_size]
            batch_paths = [paths[i] for i in batch_indices]
            batch_labels_list = [labels[i] for i in batch_indices]
            
            batch_images = open_images(batch_paths)
            batch_labels = encode_labels(batch_labels_list)
            
            yield batch_images, batch_labels

print("✅ Data generator defined successfully")

# 8. Build Model - VGG16 Transfer Learning with Fixes

In [ ]:
# Load VGG16 pre-trained on ImageNet
base_model = VGG16(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze all base model layers
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze last 4 layers for fine-tuning
base_model.layers[-2].trainable = True
base_model.layers[-3].trainable = True
base_model.layers[-4].trainable = True
base_model.layers[-5].trainable = True

trainable_count = sum(1 for l in base_model.layers if l.trainable)
frozen_count = len(base_model.layers) - trainable_count
print(f"\nVGG16 Summary:")
print(f"  Trainable layers: {trainable_count}")
print(f"  Frozen layers   : {frozen_count}")

# Build the complete model
model = Sequential([
    Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),
    base_model,
    Flatten(),
    Dropout(0.25),           # Reduced from 0.3 for better learning
    Dense(256, activation='relu'),      # Increased from 128
    Dropout(0.15),           # Reduced from 0.2
    Dense(128, activation='relu'),      # Extra layer
    Dropout(0.1),            # Additional dropout
    Dense(len(CLASS_FOLDERS), activation='softmax')
])

# Compile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['sparse_categorical_accuracy']
)

print("\n✅ Model built and compiled successfully!")
model.summary()

# 9. Train the Model

In [ ]:
# Early stopping to prevent overfitting
early_stopping = EarlyStopping(
    monitor='loss',
    patience=3,
    restore_best_weights=True
)

steps_per_epoch = int(len(train_paths) / BATCH_SIZE)
print(f"Training Configuration:")
print(f"  Total images    : {len(train_paths)}")
print(f"  Batch size      : {BATCH_SIZE}")
print(f"  Steps per epoch : {steps_per_epoch}")
print(f"  Total epochs    : {EPOCHS}")
print(f"\n🚀 Starting training...\n")

history = model.fit(
    data_generator(train_paths, train_labels, batch_size=BATCH_SIZE, epochs=EPOCHS),
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    callbacks=[early_stopping],
    verbose=1
)

print("\n✅ Training complete!")

# 10. Training History Visualization

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['sparse_categorical_accuracy'], 'o-g', linewidth=2.5, markersize=6)
plt.title('Model Accuracy Over Epochs', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.grid(True, alpha=0.3)
plt.ylim([0, 1])

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], 'o-r', linewidth=2.5, markersize=6)
plt.title('Model Loss Over Epochs', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print metrics
final_loss = history.history['loss'][-1]
final_acc = history.history['sparse_categorical_accuracy'][-1]
print(f"\n📊 Final Training Metrics:")
print(f"  Loss     : {final_loss:.4f}")
print(f"  Accuracy : {final_acc:.4f}")

# 11. Evaluate on Test Data

In [ ]:
print("Loading and evaluating on test set...\n")
test_images = open_images(test_paths)
test_labels_encoded = encode_labels(test_labels)

test_predictions = model.predict(test_images, verbose=1)
predicted_classes = np.argmax(test_predictions, axis=1)
confidence_scores = np.max(test_predictions, axis=1)

test_accuracy = np.mean(predicted_classes == test_labels_encoded)
print(f"\n{'='*70}")
print(f"✅ Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"{'='*70}")
print(f"Average Confidence: {np.mean(confidence_scores) * 100:.2f}%")
print(f"Min Confidence    : {np.min(confidence_scores) * 100:.2f}%")
print(f"Max Confidence    : {np.max(confidence_scores) * 100:.2f}%")

# 12. Classification Report

In [ ]:
print("\n" + "="*70)
print("CLASSIFICATION REPORT")
print("="*70)
print(classification_report(
    test_labels_encoded,
    predicted_classes,
    target_names=CLASS_FOLDERS
))

# 13. Confusion Matrix

In [ ]:
conf_matrix = confusion_matrix(test_labels_encoded, predicted_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(
    conf_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_FOLDERS,
    yticklabels=CLASS_FOLDERS,
    cbar_kws={'label': 'Count'}
)
plt.title("Confusion Matrix — Test Set", fontsize=13, fontweight='bold')
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

# 14. ROC Curve and AUC

In [ ]:
test_labels_bin = label_binarize(test_labels_encoded, classes=np.arange(len(CLASS_FOLDERS)))

fpr, tpr, roc_auc = {}, {}, {}
for i in range(len(CLASS_FOLDERS)):
    fpr[i], tpr[i], _ = roc_curve(test_labels_bin[:, i], test_predictions[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(10, 7))
colors_roc = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for i in range(len(CLASS_FOLDERS)):
    if i < len(colors_roc):
        plt.plot(fpr[i], tpr[i], color=colors_roc[i], linewidth=2.5,
                 label=f'{CLASS_FOLDERS[i].capitalize()}  (AUC = {roc_auc[i]:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random baseline')
plt.title("ROC Curve — One vs Rest", fontsize=13, fontweight='bold')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("AUC SCORES PER CLASS")
print("="*70)
for i, cls in enumerate(CLASS_FOLDERS):
    print(f"{cls.capitalize():15} → AUC = {roc_auc[i]:.4f}")
mean_auc = np.mean(list(roc_auc.values()))
print(f"{'Mean AUC':15} → {mean_auc:.4f}")

# 15. Save the Trained Model

In [ ]:
model.save('brain_tumour_model_fixed.h5')
print("✅ Model saved as 'brain_tumour_model_fixed.h5'")
print("\nTo download it:")
print("  1. Click Files icon (left sidebar)")
print("  2. Right-click 'brain_tumour_model_fixed.h5'")
print("  3. Click 'Download'")

# 16. Prediction Function with Fixed Class Mapping

In [ ]:
def detect_tumour(img_path, trained_model, image_size=IMAGE_SIZE):
    """
    FIXED prediction function using explicit class mapping.
    """
    # Load and preprocess
    img = load_img(img_path, target_size=(image_size, image_size))
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # Predict
    predictions = trained_model.predict(img_array, verbose=0)
    predicted_idx = np.argmax(predictions, axis=1)[0]
    confidence = np.max(predictions, axis=1)[0]
    
    # Use explicit class mapping (THE FIX!)
    predicted_class_name = IDX_TO_CLASS[predicted_idx]
    
    # Display
    plt.figure(figsize=(5, 5))
    plt.imshow(load_img(img_path))
    plt.axis('off')
    
    if predicted_class_name == 'notumor':
        title = f"✅ NO TUMOUR DETECTED\nConfidence: {confidence*100:.1f}%"
        color = 'green'
    else:
        title = f"⚠️ TUMOUR: {predicted_class_name.upper()}\nConfidence: {confidence*100:.1f}%"
        color = 'red'
    
    plt.title(title, fontsize=12, fontweight='bold', pad=10, color=color)
    plt.tight_layout()
    plt.show()
    
    return predicted_class_name, confidence

print("✅ Prediction function defined with FIXED class mapping")

# 17. Demo: Test on Sample Images

In [ ]:
print("\n" + "="*70)
print("DEMO: Testing Predictions on Sample Images")
print("="*70)

# Update these paths to your actual test images
demo_images = {}
for cls in CLASS_FOLDERS:
    cls_path = os.path.join(TEST_DIR, cls)
    if os.path.exists(cls_path):
        images = [f for f in os.listdir(cls_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
        if images:
            demo_images[cls] = os.path.join(cls_path, images[0])

if demo_images:
    for TRUE_CLASS, img_path in demo_images.items():
        print(f"\nTrue class: {TRUE_CLASS}")
        try:
            pred_class, conf = detect_tumour(img_path, model)
            status = "✅ CORRECT" if TRUE_CLASS == pred_class else "❌ WRONG"
            print(f"Predicted : {pred_class} | Confidence: {conf*100:.1f}% | {status}")
        except Exception as e:
            print(f"Error: {e}")
else:
    print("No test images found. Update the TEST_DIR path and try again.")

# 18. Manual Image Testing - Test Any Image with Custom Path

In [ ]:
# ============================================================================
# MANUAL IMAGE TESTING - Enter any image path and test it!
# ============================================================================

# Update this path to any image you want to test
# Examples:
#   '/content/drive/MyDrive/CV/MRI_Images/Testing/glioma/image.jpg'
#   '/content/drive/MyDrive/test_image.jpg'
#   'test_mri.jpg'

test_image_path = '/content/drive/MyDrive/CV/MRI_Images/Testing/glioma/Te-gl_0001.jpg'  # ← UPDATE THIS PATH

# ============================================================================

import os

# Check if file exists
if not os.path.exists(test_image_path):
    print(f"❌ Error: File not found!")
    print(f"Path: {test_image_path}")
    print(f"\nMake sure:")
    print(f"  1. The path is correct")
    print(f"  2. The file exists in Google Drive")
    print(f"  3. You have permission to access it")
else:
    print(f"✅ File found: {test_image_path}")
    print(f"\n🔍 Running prediction...\n")
    
    # Run prediction
    predicted_class, confidence = detect_tumour(test_image_path, model)
    
    # Print results
    print(f"\n{'='*70}")
    print(f"PREDICTION RESULT")
    print(f"{'='*70}")
    print(f"Image Path    : {test_image_path}")
    print(f"Predicted Class: {predicted_class.upper()}")
    print(f"Confidence     : {confidence*100:.2f}%")
    print(f"{'='*70}")
    
    # Additional details
    if predicted_class == 'notumor':
        print(f"\n✅ Result: NO TUMOUR DETECTED")
        print(f"   Patient appears to have a healthy brain scan.")
    else:
        print(f"\n⚠️  Result: TUMOUR DETECTED")
        print(f"   Type: {predicted_class.upper()}")
        print(f"   Confidence: {confidence*100:.1f}%")
        if confidence < 0.8:
            print(f"   ⚠️  Low confidence - recommend manual review")
        else:
            print(f"   ✅ High confidence prediction")

# 19. Batch Testing - Test Multiple Images at Once

In [ ]:
# ============================================================================
# BATCH TESTING - Test multiple images in a folder
# ============================================================================

# Update this to the folder containing your test images
# Examples:
#   '/content/drive/MyDrive/CV/MRI_Images/Testing/glioma/'
#   '/content/drive/MyDrive/my_mri_images/'

test_folder = '/content/drive/MyDrive/CV/MRI_Images/Testing/'  # ← UPDATE THIS PATH
num_images_to_test = 5  # How many images to test per class

# ============================================================================

import os
from PIL import Image
import numpy as np

if not os.path.exists(test_folder):
    print(f"❌ Folder not found: {test_folder}")
else:
    print(f"Testing images from: {test_folder}\n")
    
    # Get all subfolders (classes)
    classes = [d for d in os.listdir(test_folder) if os.path.isdir(os.path.join(test_folder, d))]
    
    if not classes:
        print(f"❌ No subfolders found in {test_folder}")
    else:
        total_correct = 0
        total_tested = 0
        
        for class_name in sorted(classes):
            class_folder = os.path.join(test_folder, class_name)
            images = [f for f in os.listdir(class_folder) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
            
            print(f"\n{'='*70}")
            print(f"Testing Class: {class_name.upper()}")
            print(f"Found {len(images)} images, testing {min(num_images_to_test, len(images))}")
            print(f"{'='*70}")
            
            class_correct = 0
            
            for i, img_file in enumerate(images[:num_images_to_test]):
                img_path = os.path.join(class_folder, img_file)
                
                try:
                    # Predict without showing image (for batch mode)
                    img = load_img(img_path, target_size=(IMAGE_SIZE, IMAGE_SIZE))
                    img_array = img_to_array(img) / 255.0
                    img_array = np.expand_dims(img_array, axis=0)
                    
                    predictions = model.predict(img_array, verbose=0)
                    predicted_idx = np.argmax(predictions, axis=1)[0]
                    confidence = np.max(predictions, axis=1)[0]
                    predicted_class = IDX_TO_CLASS[predicted_idx]
                    
                    is_correct = predicted_class == class_name
                    status = "✅ CORRECT" if is_correct else "❌ WRONG"
                    
                    if is_correct:
                        class_correct += 1
                    
                    print(f"  {i+1}. {img_file:30} → {predicted_class:12} ({confidence*100:5.1f}%) {status}")
                    
                except Exception as e:
                    print(f"  {i+1}. {img_file:30} → ERROR: {str(e)}")
            
            # Class summary
            class_accuracy = (class_correct / min(num_images_to_test, len(images))) * 100
            total_correct += class_correct
            total_tested += min(num_images_to_test, len(images))
            print(f"  Class Accuracy: {class_correct}/{min(num_images_to_test, len(images))} ({class_accuracy:.1f}%)")
        
        # Overall summary
        overall_accuracy = (total_correct / total_tested) * 100 if total_tested > 0 else 0
        print(f"\n{'='*70}")
        print(f"OVERALL BATCH TEST SUMMARY")
        print(f"{'='*70}")
        print(f"Total Tested : {total_tested}")
        print(f"Total Correct: {total_correct}")
        print(f"Accuracy     : {overall_accuracy:.1f}%")
        print(f"{'='*70}")